### Import Libraries

In [17]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

### Load the Dataset

In [3]:
df = pd.read_csv('../data/processed/rainfall_model_ready.csv')

In [4]:
df = df.sort_values(
    ['YEAR', 'SUBDIVISION']
).reset_index(drop=True)

In [5]:
print("Shape:", df.shape)
df.head()

Shape: (3874, 27)


,SUBDIVISION,YEAR,JAN,FEB,MAR,APR,MAY,JUN,JUL,AUG,...,Jun-Sep,Oct-Dec,annual_lag1,annual_lag2,annual_lag3,rolling_mean_3,annual_lag4,annual_lag5,rolling_mean_5,target
0,ANDAMAN & NICOBAR ISLANDS,1906,36.6,0.0,0.0,0.0,556.1,733.3,247.7,320.5,...,1465.8,475.9,2566.7,3079.6,2957.4,2867.900000,3520.7,3373.2,3099.52,3347.9
1,ASSAM & MEGHALAYA,1906,9.7,59.2,88.4,263.5,271.3,388.4,541.8,585.9,...,1780.3,190.8,2790.7,2664.2,2544.6,2666.500000,2952.1,2498.7,2690.06,2521.3
2,BIHAR,1906,14.4,56.9,8.6,0.9,44.7,191.3,366.2,430.1,...,1106.3,33.3,1498.6,1269.7,1016.4,1261.566667,1206.2,952.4,1188.66,1126.4
3,CHHATTISGARH,1906,25.0,91.0,52.5,0.0,4.1,210.1,445.2,258.3,...,1155.9,66.1,1250.9,1492.4,1331.1,1358.133333,923.9,1386.9,1277.04,1308.8
4,COASTAL ANDHRA PRADESH,1906,3.9,23.5,9.9,2.3,11.0,252.6,155.8,241.1,...,776.4,194.4,795.0,860.1,1316.1,990.400000,1063.5,993.7,1005.68,817.9


In [12]:
print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nSubdivision count:")
print(df["SUBDIVISION"].nunique())

print("\nRows per subdivision:")
print(df["SUBDIVISION"].value_counts().sort_index())

print("\nYear range:")
print(df["YEAR"].min(), "to", df["YEAR"].max())

Dataset shape: (3874, 27)

Columns:
['SUBDIVISION', 'YEAR', 'JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN', 'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC', 'ANNUAL', 'Jan-Feb', 'Mar-May', 'Jun-Sep', 'Oct-Dec', 'annual_lag1', 'annual_lag2', 'annual_lag3', 'rolling_mean_3', 'annual_lag4', 'annual_lag5', 'rolling_mean_5', 'target']

Subdivision count:
36

Rows per subdivision:
SUBDIVISION
ANDAMAN & NICOBAR ISLANDS              90
ARUNACHAL PRADESH                      85
ASSAM & MEGHALAYA                     109
BIHAR                                 109
CHHATTISGARH                          109
COASTAL ANDHRA PRADESH                109
COASTAL KARNATAKA                     109
EAST MADHYA PRADESH                   109
EAST RAJASTHAN                        109
EAST UTTAR PRADESH                    109
GANGETIC WEST BENGAL                  109
GUJARAT REGION                        109
HARYANA DELHI & CHANDIGARH            109
HIMACHAL PRADESH                      109
JAMMU & KASHMIR                    

In [13]:
print("Target column exists:", "target" in df.columns)

required_features = [
    "annual_lag1",
    "annual_lag2",
    "annual_lag3",
    "annual_lag4",
    "annual_lag5",
    "rolling_mean_3",
    "rolling_mean_5",
]

print("\nMissing required features:")
print([col for col in required_features if col not in df.columns])

print("\nMissing values in required columns:")
print(df[required_features + ["target"]].isna().sum())

Target column exists: True

Missing required features:
[]

Missing values in required columns:
annual_lag1       0
annual_lag2       0
annual_lag3       0
annual_lag4       0
annual_lag5       0
rolling_mean_3    0
rolling_mean_5    0
target            0
dtype: int64


### Define Features

### Train Test Split

In [15]:
# Version 1 numeric features
model_features = [
    "annual_lag1",
    "annual_lag2",
    "annual_lag3",
    "annual_lag4",
    "annual_lag5",
    "rolling_mean_3",
    "rolling_mean_5",
]

target_column = "target"

# Chronological split based on the feature year
train_data = df[df["YEAR"] <= 2000].copy()

validation_data = df[
    (df["YEAR"] >= 2001) & (df["YEAR"] <= 2008)
].copy()

holdout_data = df[
    (df["YEAR"] >= 2009) & (df["YEAR"] <= 2014)
].copy()

In [16]:
print("Training rows:", len(train_data))
print("Validation rows:", len(validation_data))
print("Holdout rows:", len(holdout_data))

print("\nTraining year range:",
      train_data["YEAR"].min(), "-", train_data["YEAR"].max())

print("Validation year range:",
      validation_data["YEAR"].min(), "-", validation_data["YEAR"].max())

print("Holdout year range:",
      holdout_data["YEAR"].min(), "-", holdout_data["YEAR"].max())

Training rows: 3370
Validation rows: 288
Holdout rows: 216

Training year range: 1906 - 2000
Validation year range: 2001 - 2008
Holdout year range: 2009 - 2014


### Model A- without subdivisions

In [18]:
X_train = train_data[model_features]
y_train = train_data[target_column]

X_val = validation_data[model_features]
y_val = validation_data[target_column]

model_a = LinearRegression()
model_a.fit(X_train, y_train)

pred_a = model_a.predict(X_val)

print("Model A — Original Linear Regression")
print("MAE:", mean_absolute_error(y_val, pred_a))
print("RMSE:", np.sqrt(mean_squared_error(y_val, pred_a)))
print("R²:", r2_score(y_val, pred_a))

Model A — Original Linear Regression
MAE: 224.26198826661027
RMSE: 291.2273444707588
R²: 0.8775979047381097


## Model B- with subdivisions

In [19]:
numeric_features = model_features
categorical_features = ["SUBDIVISION"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "location",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

model_b = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

X_train_b = train_data[numeric_features + categorical_features]
X_val_b = validation_data[numeric_features + categorical_features]

model_b.fit(X_train_b, y_train)

pred_b = model_b.predict(X_val_b)

print("Model B — Location-Aware Linear Regression")
print("MAE:", mean_absolute_error(y_val, pred_b))
print("RMSE:", np.sqrt(mean_squared_error(y_val, pred_b)))
print("R²:", r2_score(y_val, pred_b))

Model B — Location-Aware Linear Regression
MAE: 222.55673231705623
RMSE: 287.9897801047198
R²: 0.880304257200926


In [20]:
# Prepare holdout data
X_holdout = holdout_data[model_features]
y_holdout = holdout_data["target"]

X_holdout_b = holdout_data[model_features + ["SUBDIVISION"]]

# Predictions from the already-trained models
pred_a_holdout = model_a.predict(X_holdout)
pred_b_holdout = model_b.predict(X_holdout_b)

# Compare metrics
results = pd.DataFrame({
    "Model": ["Model A - No subdivision",
              "Model B - With subdivision"],
    "MAE": [
        mean_absolute_error(y_holdout, pred_a_holdout),
        mean_absolute_error(y_holdout, pred_b_holdout)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_holdout, pred_a_holdout)),
        np.sqrt(mean_squared_error(y_holdout, pred_b_holdout))
    ],
    "R2": [
        r2_score(y_holdout, pred_a_holdout),
        r2_score(y_holdout, pred_b_holdout)
    ]
})

results

,Model,MAE,RMSE,R2
0,Model A - No subdivision,211.839813,289.060999,0.891040
1,Model B - With subdivision,213.135850,288.406531,0.891533


In [21]:
# Create a comparison table for each holdout observation
comparison = holdout_data[
    ["SUBDIVISION", "YEAR", "target"]
].copy()

comparison["pred_model_a"] = pred_a_holdout
comparison["pred_model_b"] = pred_b_holdout

comparison["abs_error_a"] = (
    comparison["target"] - comparison["pred_model_a"]
).abs()

comparison["abs_error_b"] = (
    comparison["target"] - comparison["pred_model_b"]
).abs()

# Average absolute error for each subdivision
regional_results = (
    comparison
    .groupby("SUBDIVISION")
    .agg(
        MAE_A=("abs_error_a", "mean"),
        MAE_B=("abs_error_b", "mean"),
        observations=("target", "size")
    )
)

regional_results["MAE_change"] = (
    regional_results["MAE_B"] - regional_results["MAE_A"]
)

regional_results = regional_results.sort_values("MAE_change")

regional_results

,MAE_A,MAE_B,observations,MAE_change
SUBDIVISION,,,,
ANDAMAN & NICOBAR ISLANDS,629.250926,528.395313,6,-100.855613
LAKSHADWEEP,200.054276,144.592175,6,-55.462101
CHHATTISGARH,134.008889,80.982453,6,-53.026435
SAURASHTRA & KUTCH,276.712771,225.182803,6,-51.529968
JAMMU & KASHMIR,256.490636,213.411999,6,-43.078637
TAMIL NADU,216.970871,176.247518,6,-40.723353
MADHYA MAHARASHTRA,162.231905,129.146823,6,-33.085082
NORTH INTERIOR KARNATAKA,135.322890,110.306812,6,-25.016079
COASTAL KARNATAKA,436.256002,416.891860,6,-19.364142


In [22]:
validation_comparison = validation_data[
    ["SUBDIVISION", "YEAR", "target"]
].copy()

validation_comparison["pred_a"] = pred_a
validation_comparison["pred_b"] = pred_b

validation_comparison["error_a"] = (
    validation_comparison["target"] - validation_comparison["pred_a"]
).abs()

validation_comparison["error_b"] = (
    validation_comparison["target"] - validation_comparison["pred_b"]
).abs()

validation_regional = (
    validation_comparison
    .groupby("SUBDIVISION")
    .agg(
        MAE_A=("error_a", "mean"),
        MAE_B=("error_b", "mean"),
        observations=("target", "size")
    )
)

validation_regional["MAE_change"] = (
    validation_regional["MAE_B"] - validation_regional["MAE_A"]
)

validation_regional = validation_regional.sort_values("MAE_change")

print("Regions improved on validation:",
      (validation_regional["MAE_change"] < 0).sum())

print("Regions worsened on validation:",
      (validation_regional["MAE_change"] > 0).sum())

validation_regional

Regions improved on validation: 26
Regions worsened on validation: 10


,MAE_A,MAE_B,observations,MAE_change
SUBDIVISION,,,,
TAMIL NADU,385.501643,313.497716,8,-72.003928
KONKAN & GOA,424.037556,353.604238,8,-70.433318
LAKSHADWEEP,311.545134,242.129346,8,-69.415789
KERALA,430.802892,374.664436,8,-56.138457
COASTAL KARNATAKA,441.858255,387.049333,8,-54.808922
GUJARAT REGION,347.848203,297.025104,8,-50.823099
JAMMU & KASHMIR,289.440057,250.412507,8,-39.027550
MADHYA MAHARASHTRA,226.795539,189.475775,8,-37.319764
NORTH INTERIOR KARNATAKA,162.448532,131.411883,8,-31.036649


## Model C- Random Forest with subdivisions

In [23]:
from sklearn.ensemble import RandomForestRegressor

preprocessor_c = ColumnTransformer(
    transformers=[
        (
            "location",
            OneHotEncoder(handle_unknown="ignore"),
            ["SUBDIVISION"]
        )
    ],
    remainder="passthrough"
)

model_c = Pipeline([
    ("preprocessor", preprocessor_c),
    ("regressor", RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ))
])

X_train_c = train_data[model_features + ["SUBDIVISION"]]
X_val_c = validation_data[model_features + ["SUBDIVISION"]]

model_c.fit(X_train_c, y_train)
pred_c = model_c.predict(X_val_c)

print("Model C — Location-Aware Random Forest")
print("MAE:", mean_absolute_error(y_val, pred_c))
print("RMSE:", np.sqrt(mean_squared_error(y_val, pred_c)))
print("R²:", r2_score(y_val, pred_c))

Model C — Location-Aware Random Forest
MAE: 223.59381135036
RMSE: 284.8730055311556
R²: 0.8828810562675877


In [24]:
validation_results = pd.DataFrame({
    "Model": [
        "A - Linear Regression",
        "B - Linear Regression + Subdivision",
        "C - Random Forest + Subdivision"
    ],
    "MAE": [
        mean_absolute_error(y_val, pred_a),
        mean_absolute_error(y_val, pred_b),
        mean_absolute_error(y_val, pred_c)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_val, pred_a)),
        np.sqrt(mean_squared_error(y_val, pred_b)),
        np.sqrt(mean_squared_error(y_val, pred_c))
    ],
    "R2": [
        r2_score(y_val, pred_a),
        r2_score(y_val, pred_b),
        r2_score(y_val, pred_c)
    ]
})

validation_results.sort_values("MAE")

,Model,MAE,RMSE,R2
1,B - Linear Regression + Subdivision,222.556732,287.989780,0.880304
2,C - Random Forest + Subdivision,223.593811,284.873006,0.882881
0,A - Linear Regression,224.261988,291.227344,0.877598


### Train Final Linear Regression Model

In [11]:
final_model = LinearRegression()

final_model.fit(
    X_train,
    y_train
)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](7,)","[0.22,0.14,0.05,...,0.17,0.14,0.14]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](7,)","['annual_lag1','annual_lag2','annual_lag3',...,'annual_lag5', 'rolling_mean_3','rolling_mean_5']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,42.81
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,7
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(5)


### Prediction and Evaluation

In [11]:
validation_pred = final_model.predict(
    X_validation
)

validation_mae = mean_absolute_error(
    y_validation,
    validation_pred
)

validation_rmse = np.sqrt(
    mean_squared_error(
        y_validation,
        validation_pred
    )
)

validation_r2 = r2_score(
    y_validation,
    validation_pred
)

print("Validation MAE:", validation_mae)
print("Validation RMSE:", validation_rmse)
print("Validation R²:", validation_r2)

Validation MAE: 224.26198826661027
Validation RMSE: 291.2273444707588
Validation R²: 0.8775979047381097


### Final Holdout Evaluation

In [12]:
holdout_pred = final_model.predict(
    X_holdout
)

In [13]:
holdout_mae = mean_absolute_error(
    y_holdout,
    holdout_pred
)

holdout_rmse = np.sqrt(
    mean_squared_error(
        y_holdout,
        holdout_pred
    )
)

holdout_r2 = r2_score(
    y_holdout,
    holdout_pred
)

print("Final Holdout MAE:", holdout_mae)
print("Final Holdout RMSE:", holdout_rmse)
print("Final Holdout R²:", holdout_r2)

Final Holdout MAE: 211.83981251906286
Final Holdout RMSE: 289.06099919324186
Final Holdout R²: 0.8910400567984633


In [14]:
import joblib

joblib.dump(
    final_model,
    '../models/linear_regression_rainfall.joblib'
)

print("Model saved successfully.")

Model saved successfully.


In [15]:
import os

model_path = '../models/linear_regression_rainfall.joblib'

print("Model exists:", os.path.exists(model_path))

Model exists: True


In [16]:
import sys
print(sys.executable)

C:\Users\HP\OneDrive\Desktop\Rainfall_Prediction\.venv\Scripts\python.exe


In [17]:
import sklearn
print("sklearn:", sklearn.__version__)

sklearn: 1.9.1


In [25]:
import joblib
from pathlib import Path

# Combine training and validation data
development_data = pd.concat(
    [train_data, validation_data],
    ignore_index=True
)

X_development = development_data[
    model_features + ["SUBDIVISION"]
]
y_development = development_data["target"]

# Train the selected location-aware pipeline
final_model = Pipeline([
    ("preprocessor", ColumnTransformer(
        transformers=[
            (
                "location",
                OneHotEncoder(handle_unknown="ignore"),
                ["SUBDIVISION"]
            )
        ],
        remainder="passthrough"
    )),
    ("regressor", LinearRegression())
])

final_model.fit(X_development, y_development)

# Save as a separate model file; preserve Version 1
model_path = Path("../models/linear_regression_subdivision.joblib")
model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(final_model, model_path)

print("Final model saved to:", model_path)
print("Training rows:", len(development_data))
print("Training year range:",
      development_data["YEAR"].min(),
      "-",
      development_data["YEAR"].max())

Final model saved to: ..\models\linear_regression_subdivision.joblib
Training rows: 3658
Training year range: 1906 - 2008
